In [ ]:
%cd ../..

import os
import torch
from tqdm import tqdm
from omegaconf import OmegaConf
from glob import glob
import numpy as np
import random
from math import pi
import pandas as pd
import SimpleITK as sitk

from evaluation import *

from dinov2.inference import generate_embeddings, build_model

In [ ]:
def prep_object_detection(path, posx, posy, posz, pad_range=(112,448)):
    image_obj = sitk.ReadImage(path)
    image = sitk.GetArrayFromImage(image_obj)
    image = image.clip(-1000, 1900)

    pad = random.randint(*pad_range)

    D, W, H = image.shape
    
    origin = np.array(image_obj.GetOrigin())[::-1]
    spacing = np.array(image_obj.GetSpacing())[::-1]

    direction_flat = image_obj.GetDirection()
    dim = image_obj.GetDimension()
    direction = torch.tensor(direction_flat).reshape((dim, dim))

    assert torch.all(direction * torch.eye(dim) == direction)

    dz = direction[2][2]
    dy = direction[1][1]
    dx = direction[0][0]

    coordz = int(dz * (posz - origin[0])/spacing[0])
    coordy = int(dy *(posy - origin[1])/spacing[1])
    coordx = int(dx * (posx - origin[2])/spacing[2])

    xmin_min = max(0, coordx - pad + 1)
    xmin_max = min(coordx, H - pad)
    xmin = random.randint(xmin_min, xmin_max)
    xmax = xmin + pad

    ymin_min = max(0, coordy - pad + 1)
    ymin_max = min(coordy, W - pad)
    ymin = random.randint(ymin_min, ymin_max)
    ymax = ymin + pad

    z_radius = 3
    zmin = max(0, coordz - z_radius)
    zmax = min(D, coordz + z_radius)

    slice_obj = (slice(zmin, zmax), slice(ymin, ymax), slice(xmin, xmax))
    cropped_img = image[slice_obj]
    cropped_img = torch.from_numpy(cropped_img).float()

    relx = (coordx - xmin) / (xmax - xmin)
    rely = (coordy - ymin) / (ymax - ymin)
    rel_pos = torch.tensor([relx, rely])

    k = random.randint(0,3)
    theta = -k * pi / 2
    c, s = torch.cos(torch.tensor(theta)), torch.sin(torch.tensor(theta))
    R = torch.tensor([[c, -s], [s,  c]])
    
    rel_pos = (rel_pos - 0.5) @ R.T + 0.5

    cropped_img = torch.rot90(cropped_img, k=k, dims=(1,2))
    
    return cropped_img, rel_pos


In [ ]:
dataset_root = "/data/work/vm/radio-foundation/LUNA16"
img_paths = glob(os.path.join(dataset_root,"subset*/**/*.mhd"), recursive=True)
id_to_path = {p.split("/")[-1].replace(".mhd", ""): p for p in img_paths}
len(img_paths)

In [ ]:
candidates_df = pd.read_csv(os.path.join(dataset_root, "candidates.csv"))
candidatesv2_df = pd.read_csv(os.path.join(dataset_root, "candidates_V2.csv"))
annotations_df = pd.read_csv(os.path.join(dataset_root, "annotations.csv"))
annotations_df

In [ ]:
max_rows = 1186 * 5
candidatesv2_df = candidatesv2_df[candidatesv2_df["class"] == 0]
candidatesv2_df = candidatesv2_df.sample(n=max_rows, random_state=42)

candidatesv2_df

In [ ]:
seriesuid, coordX, coordY, coordZ, diameter = annotations_df.iloc[0]
path = id_to_path[seriesuid]
diameter

In [ ]:
seriesuid, coordX, coordY, coordZ, _ = candidatesv2_df.iloc[19]
img_path = id_to_path[seriesuid]

In [ ]:
img, rel_pos = prep_object_detection(img_path, coordX, coordY, coordZ)
posx, posy = rel_pos
view_object(img, posx, posy)

In [ ]:
config_path = "/home/48078029W/projects/radio-foundation/runs/base10pat/config.yaml"
checkpoint_path = "/home/48078029W/projects/radio-foundation/runs/base10pat/eval/training_99999/teacher_checkpoint.pth"

device = torch.device("cuda")

config = OmegaConf.load(config_path)
model, autocast_ctx = build_model(checkpoint_path, config, img_size=504, device=device)

data_kwargs = dict(
    fmean = -573.8,
    fstd = 461.3,
    channels = 10,
    img_size = 504,
    patch_size = 14,
    device="cuda",
    block_size=64,
    autocast_ctx=autocast_ctx,
    no_crop=True
)

In [ ]:
output_path_train = "/data/work/vm/radio-foundation/embeddings/LUNA16/detection/train"
output_path_val = "/data/work/vm/radio-foundation/embeddings/LUNA16/detection/val"
os.makedirs(output_path_train, exist_ok=True)
os.makedirs(output_path_val, exist_ok=True)

num_ids = len(annotations_df)
num_train = int(num_ids * 0.8)

ids_list = list(range(num_ids))
random.shuffle(ids_list)
train_ids = ids_list[:num_train]
val_ids = ids_list[num_train:]

for idx, row in tqdm(annotations_df.iterrows(), total=num_ids):
    seriesuid, coordX, coordY, coordZ, diameter = row
    img_path = id_to_path[seriesuid]
    
    output = {}

    for rep_idx in range(10):

        img, rel_pos = prep_object_detection(img_path, coordX, coordY, coordZ)
        img = torch.nn.functional.interpolate(
            img.unsqueeze(0).unsqueeze(0), size=(data_kwargs["channels"], data_kwargs["img_size"], data_kwargs["img_size"]), mode="trilinear"
        ).squeeze(0).squeeze(0)

        collated_features = generate_embeddings(
            img,
            model=model,
            **data_kwargs # type: ignore
        )

        output[f"patch_{rep_idx:02}"] = collated_features["patch"]
        output[f"pos_{rep_idx:02}"] = rel_pos

    if idx in train_ids:
        output_path = output_path_train
    else:
        output_path = output_path_val

    torch.save(output, os.path.join(output_path, f"{idx:06}.pth"))
